### Вспомогательные функции

In [ ]:
# Аналог \\w без регулярных выражений: буквы, цифры и подчёркивание.
def is_word_char(ch):
    return ch.isalnum() or ch == "_"

# Ручная токенизация по словам:
#     - последовательности букв/цифр/_ считаются одним токеном
#     - знаки пунктуации считаются отдельными токенами
#     - пробелы пропускаются
def manual_word_tokenize(text, lowercase=True):
    if lowercase:
        text = text.lower()

    tokens = []
    current = []

    for ch in text:
        if is_word_char(ch):
            current.append(ch)
        else:
            if current:
                tokens.append("".join(current))
                current = []

            if not ch.isspace():
                tokens.append(ch)

    if current:
        tokens.append("".join(current))

    return tokens

### Символьная токенизация

In [ ]:
class CharTokenizer:
    def __init__(self, special_tokens=None):
        if special_tokens is None:
            special_tokens = ["<PAD>", "<UNK>"]

        self.special_tokens = special_tokens
        self.token_to_id = {}
        self.id_to_token = {}

    def fit(self, texts):
        chars = {}

        for text in texts:
            for ch in text:
                chars[ch] = True

        vocab = []

        for tok in self.special_tokens:
            if tok not in vocab:
                vocab.append(tok)

        for ch in sorted(chars.keys()):
            if ch not in vocab:
                vocab.append(ch)

        self.token_to_id = {}

        for i, tok in enumerate(vocab):
            self.token_to_id[tok] = i

        self.id_to_token = {}

        for tok, idx in self.token_to_id.items():
            self.id_to_token[idx] = tok

    def encode(self, text):
        unk_id = self.token_to_id["<UNK>"]

        ids = []

        for ch in text:
            ids.append(self.token_to_id.get(ch, unk_id))

        return ids

    def decode(self, ids):
        tokens = []

        for idx in ids:
            tokens.append(self.id_to_token.get(idx, "<UNK>"))

        return "".join(tokens)

    @property
    def vocab_size(self):
        return len(self.token_to_id)

### Токенизация по словам

In [ ]:
class WordTokenizer:
    def __init__(self, special_tokens=None, max_vocab_size=None, lowercase=True):
        if special_tokens is None:
            special_tokens = ["<PAD>", "<UNK>"]

        self.special_tokens = special_tokens
        self.max_vocab_size = max_vocab_size
        self.lowercase = lowercase

        self.token_to_id = {}
        self.id_to_token = {}

    def _tokenize(self, text):
        return manual_word_tokenize(text, lowercase=self.lowercase)

    def fit(self, texts):
        counts = {}
        first_seen = {}
        position = 0

        for text in texts:
            tokens = self._tokenize(text)

            for tok in tokens:
                if tok not in counts:
                    counts[tok] = 0
                    first_seen[tok] = position
                    position += 1

                counts[tok] += 1

        sorted_tokens = sorted(
            counts.keys(),
            key=lambda tok: (-counts[tok], first_seen[tok])
        )

        if self.max_vocab_size is not None:
            sorted_tokens = sorted_tokens[:self.max_vocab_size]

        vocab = []

        for tok in self.special_tokens:
            if tok not in vocab:
                vocab.append(tok)

        for tok in sorted_tokens:
            if tok not in vocab:
                vocab.append(tok)

        self.token_to_id = {}

        for i, tok in enumerate(vocab):
            self.token_to_id[tok] = i

        self.id_to_token = {}

        for tok, idx in self.token_to_id.items():
            self.id_to_token[idx] = tok

    def encode(self, text):
        unk_id = self.token_to_id["<UNK>"]

        ids = []

        for tok in self._tokenize(text):
            ids.append(self.token_to_id.get(tok, unk_id))

        return ids

    def decode(self, ids):
        tokens = []

        for idx in ids:
            tokens.append(self.id_to_token.get(idx, "<UNK>"))

        return " ".join(tokens)

    @property
    def vocab_size(self):
        return len(self.token_to_id)

### BPE-токенизация

In [ ]:
import json


class SimpleBPETokenizer:
    END_WORD = "</w>"

    def __init__(
        self,
        vocab_size=8000,
        special_tokens=None,
        lowercase=True
    ):
        if special_tokens is None:
            special_tokens = ["<PAD>", "<UNK>", "<BOS>", "<EOS>"]

        self.vocab_size_limit = vocab_size
        self.special_tokens = special_tokens
        self.lowercase = lowercase

        self.merges = []
        self.token_to_id = {}
        self.id_to_token = {}

    def _tokenize_words(self, text):
        return manual_word_tokenize(text, lowercase=self.lowercase)

    # Объединяет все вхождения пары pair в последовательности symbols
    def _merge_sequence(self, symbols, pair, new_token):
        result = []
        i = 0

        while i < len(symbols):
            if (
                i < len(symbols) - 1
                and symbols[i] == pair[0]
                and symbols[i + 1] == pair[1]
            ):
                result.append(new_token)
                i += 2
            else:
                result.append(symbols[i])
                i += 1

        return result

    # Применяет выученные BPE-слияния к одному слову
    def _apply_merges(self, symbols):
        for pair in self.merges:
            new_token = pair[0] + pair[1]
            symbols = self._merge_sequence(symbols, pair, new_token)

        return symbols

    def fit(self, texts):
        word_freq = {}
        alphabet = {}

        for text in texts:
            words = self._tokenize_words(text)

            for word in words:
                if word not in word_freq:
                    word_freq[word] = 0

                word_freq[word] += 1

                for ch in word:
                    alphabet[ch] = True

        corpus = {}

        for word, freq in word_freq.items():
            symbols = tuple(list(word) + [self.END_WORD])
            corpus[symbols] = freq

        vocab = []

        def add_token(tok):
            if tok not in vocab:
                vocab.append(tok)

        for tok in self.special_tokens:
            add_token(tok)

        for ch in sorted(alphabet.keys()):
            add_token(ch)

        add_token(self.END_WORD)

        while len(vocab) < self.vocab_size_limit:
            pair_counts = {}
            pair_first_seen = {}
            pair_position = 0

            for symbols, freq in corpus.items():
                if len(symbols) < 2:
                    continue

                for i in range(len(symbols) - 1):
                    pair = (symbols[i], symbols[i + 1])

                    if pair not in pair_counts:
                        pair_counts[pair] = 0
                        pair_first_seen[pair] = pair_position
                        pair_position += 1

                    pair_counts[pair] += freq

            if not pair_counts:
                break

            best_pair = None
            best_count = -1
            best_position = None

            for pair, count in pair_counts.items():
                position = pair_first_seen[pair]

                if (
                    count > best_count
                    or count == best_count and position < best_position
                ):
                    best_pair = pair
                    best_count = count
                    best_position = position

            if best_pair is None:
                break

            new_token = best_pair[0] + best_pair[1]

            self.merges.append(best_pair)
            add_token(new_token)

            new_corpus = {}

            for symbols, freq in corpus.items():
                merged_symbols = self._merge_sequence(
                    list(symbols),
                    best_pair,
                    new_token
                )

                merged_symbols = tuple(merged_symbols)

                if merged_symbols not in new_corpus:
                    new_corpus[merged_symbols] = 0

                new_corpus[merged_symbols] += freq

            corpus = new_corpus

        self.token_to_id = {}

        for i, tok in enumerate(vocab):
            self.token_to_id[tok] = i

        self.id_to_token = {}

        for tok, idx in self.token_to_id.items():
            self.id_to_token[idx] = tok

    def encode(self, text):
        unk_id = self.token_to_id["<UNK>"]

        ids = []

        words = self._tokenize_words(text)

        for word in words:
            symbols = []

            for ch in word:
                if ch in self.token_to_id:
                    symbols.append(ch)
                else:
                    symbols.append("<UNK>")

            symbols.append(self.END_WORD)

            symbols = self._apply_merges(symbols)

            for tok in symbols:
                ids.append(self.token_to_id.get(tok, unk_id))

        return ids

    def decode(self, ids):
        words = []
        current_word = ""

        for idx in ids:
            tok = self.id_to_token.get(idx, "<UNK>")

            if tok in self.special_tokens and tok != "<UNK>":
                continue

            if tok == self.END_WORD:
                if current_word:
                    words.append(current_word)
                    current_word = ""
                continue

            if tok.endswith(self.END_WORD):
                piece = tok[:-len(self.END_WORD)]
                current_word += piece

                if current_word:
                    words.append(current_word)
                    current_word = ""

                continue

            current_word += tok

        if current_word:
            words.append(current_word)

        return " ".join(words)

    def get_vocab_size(self):
        return len(self.token_to_id)

    @property
    def vocab_size(self):
        return len(self.token_to_id)

    def save(self, path):
        data = {
            "vocab_size_limit": self.vocab_size_limit,
            "special_tokens": self.special_tokens,
            "lowercase": self.lowercase,
            "merges": self.merges,
            "token_to_id": self.token_to_id
        }

        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        tokenizer = cls(
            vocab_size=data["vocab_size_limit"],
            special_tokens=data["special_tokens"],
            lowercase=data["lowercase"]
        )

        tokenizer.merges = []

        for pair in data["merges"]:
            tokenizer.merges.append(tuple(pair))

        tokenizer.token_to_id = {}

        for tok, idx in data["token_to_id"].items():
            tokenizer.token_to_id[tok] = int(idx)

        tokenizer.id_to_token = {}

        for tok, idx in tokenizer.token_to_id.items():
            tokenizer.id_to_token[idx] = tok

        return tokenizer